## 21. 生产化：把 agent 变成可运维的服务

> 来源：[Hosting the Agent SDK](https://code.claude.com/docs/en/agent-sdk/hosting)、[Track cost and usage](https://code.claude.com/docs/en/agent-sdk/cost-tracking)、[Observability with OpenTelemetry](https://code.claude.com/docs/en/agent-sdk/observability)、[Rewind file changes with checkpointing](https://code.claude.com/docs/en/agent-sdk/file-checkpointing)、[Securely deploying AI agents](https://code.claude.com/docs/en/agent-sdk/secure-deployment)、[Migrate to Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk/migration-guide)


### 21.1 错误处理

SDK 层的异常类型（从 `claude_agent_sdk` 导入）：

```python
try:
    async for message in query(prompt="Hello"):
        print(message)
except CLINotFoundError:      # CLI 二进制找不到，重装 SDK
    ...
except ProcessError as e:     # 子进程非零退出，e.exit_code
    ...
except CLIJSONDecodeError:    # stdout 解析失败
    ...
```

再叠加两条已讲过的规则：业务级失败看 `ResultMessage.subtype`（§7.3「ResultMessage subtype 全表」）；`query()` 在 yield 错误 result 后还会 raise（§4.2「两种输入模式」）。超时与重试由 `env` 里的变量控制：`API_TIMEOUT_MS`（单请求超时，默认 600000）、`CLAUDE_CODE_MAX_RETRIES`（默认 10）、`CLAUDE_ASYNC_AGENT_STALL_TIMEOUT_MS`（后台子 agent 卡死看门狗）。


### 21.2 成本追踪

先看一条真实 `ResultMessage` 上的账目（§7.3「ResultMessage subtype 全表」 那次 poem 会话，主 agent 派了一个子 agent）：

```python
ResultMessage(
    total_cost_usd=2.4214775,                       # 客户端估算，不是账单
    usage={'input_tokens': 2922,
           'cache_creation_input_tokens': 172539,   # 写缓存（费率更高）
           'cache_read_input_tokens': 0,            # 读缓存（便宜）——本次全是首写，没省到
           'output_tokens': 542, ...},              # 只计主对话的输出
    model_usage={'claude-fable-5': {
        'inputTokens': 2924, 'outputTokens': 1405,  # 1405 = 主对话 542 + 子 agent 863，
        'cacheCreationInputTokens': 185759,         # model_usage 把子 agent 的消耗也计入
        'costUSD': 2.4214775, 'contextWindow': 200000, ...}},
)
```

围绕这几个字段的规则：

- **`total_cost_usd` 是客户端估算，不是账单**——SDK 用打包时内置的价格表本地计算，模型/价格变动时会漂移。开发调试时看个大概、做粗略预算，用它就够；正式对账走 Usage and Cost API。`max_budget_usd` 拿它做熔断。
- 三个统计口径：**step**（一次请求/响应，`AssistantMessage.usage` + `message_id`）、**query() 调用**（一个 `ResultMessage`，`total_cost_usd` 为该次调用累计）、**session**（多次 query() 串联，SDK 不提供合计，自己累加各次 result）。
- **并行 tool call 的多条 `AssistantMessage` 共享同一个 `message_id`、usage 相同**——按 step 统计要按 ID 去重，否则重复计数。罕见情况下同 ID 消息的 `output_tokens` 不一致：取最大值（组内最后一条通常是准确总数），且优先信 result 上的 `total_cost_usd`（SDK 的累计比自己逐 step 求和可靠）。
- 错误 result 同样带 `usage` / `total_cost_usd`（token 已经花了），失败也要记账。
- prompt cache 自动启用，无需配置；`cache_creation_input_tokens` 和 `cache_read_input_tokens` 分开看节省（上面实例里 172K 全是首写、零读取——单次会话吃不到 cache 红利，多轮才吃得到）。短会话间隔超过 5 分钟导致 cache 反复过期时，设 `env={"ENABLE_PROMPT_CACHING_1H": "1"}` 换 1 小时 TTL（写入费率更高；Claude 订阅用户已自动享有 1 小时 TTL，无需设置）。
- 按模型拆分看 `model_usage`（主 agent 用大模型、子 agent 用小模型时看 token 流向；注意它聚合了子 agent 的消耗，见上方实例）。
- 量级感：**token 成本通常比容器基础设施高一个数量级以上**——最小配置的容器约 $0.05/小时，而一次长 agent 会话的 token 就能花掉几美元。容量规划先算 token，再算机器（§21.6「部署」）。

### 21.3 可观测性：OpenTelemetry

CLI 内建 OTel 埋点（SDK 自己不产遥测，只透传配置），三种信号独立开关，导出到任何 OTLP 后端（Grafana、Datadog、Langfuse 等）：

```python
OTEL_ENV = {
    "CLAUDE_CODE_ENABLE_TELEMETRY": "1",
    "CLAUDE_CODE_ENHANCED_TELEMETRY_BETA": "1",   # 仅 traces 需要（traces 仍是 beta，span 名可能变）
    "OTEL_TRACES_EXPORTER": "otlp",
    "OTEL_METRICS_EXPORTER": "otlp",
    "OTEL_LOGS_EXPORTER": "otlp",
    "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
    "OTEL_EXPORTER_OTLP_ENDPOINT": "http://collector.example.com:4318",
    "OTEL_EXPORTER_OTLP_HEADERS": "Authorization=Bearer your-token",  # 托管后端的认证头
}
options = ClaudeAgentOptions(env=OTEL_ENV)  # 或直接设在容器环境里
```

导出节奏与丢数据风险：metrics 默认每 60 秒、traces/logs 默认每 5 秒批量导出——进程被 kill 时缓冲区里的 batch 会丢，clean exit 的 flush 也受短超时限制。短生命周期进程把 `OTEL_METRIC_EXPORT_INTERVAL` / `OTEL_BSP_SCHEDULE_DELAY`（traces）/ `OTEL_BLRP_SCHEDULE_DELAY`（logs）调到 1000ms。

span 与归属：

- span 结构 `interaction → llm_request / tool → tool.execution`，另有 `tool.blocked_on_user`（等权限审批的时间单独成 span）；hook 执行的 span 需另设 `ENABLE_BETA_TRACING_DETAILED=1`。子 agent 的 spans 嵌套在父 agent 的 `tool` span 下，形成完整委派链。
- spans 带 `session.id` 属性，可把多次 query() 串成一条时间线（不想带就把 `OTEL_METRICS_INCLUDE_SESSION_ID` 设为 falsy）。
- 应用侧已有活跃 span 时 SDK 自动注入 `TRACEPARENT`，agent trace 挂在你的应用 trace 下；在 `env` 里显式设 `TRACEPARENT` 则跳过自动注入（可钉住指定父 context）。CLI 还会把它转发给每条 Bash 命令，命令自身的 spans 嵌套在 `tool.execution` 下。
- 多 agent 用 `OTEL_SERVICE_NAME` / `OTEL_RESOURCE_ATTRIBUTES` 区分；塞 `enduser.id` / `tenant.id` 做逐用户审计——值必须百分号编码（逗号、空格、等号是保留字符）。

隐私默认与 opt-in：**prompt 与 tool 内容默认不导出**。逐档放开：`OTEL_LOG_USER_PROMPTS=1`（prompt 文本）、`OTEL_LOG_TOOL_DETAILS=1`（工具名与参数）、`OTEL_LOG_TOOL_CONTENT=1`（工具输入输出全文进 span events，60KB 截断）、`OTEL_LOG_RAW_API_BODIES`（完整 API 请求/响应 JSON，`1`=内联 60KB 截断、`file:<dir>`=不截断落盘；body 含全部对话历史——开它等于默许前面几档会暴露的一切）。

最后一条：**不要用 `console` exporter**——stdout 是 SDK 的消息通道。

### 21.4 文件快照与任务清单

**File checkpointing** 是四步机制，代码见下方 cell。

In [ ]:
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    ResultMessage,
    UserMessage,
)

options = ClaudeAgentOptions(
    enable_file_checkpointing=True,               # ① 开启文件快照
    extra_args={"replay-user-messages": None},    # ② 让 UserMessage.uuid 出现在流里，
)                                                 #    它就是 checkpoint ID

checkpoints = []
session_id = None
async with ClaudeSDKClient(options=options) as client:
    await client.query("重构 utils.py，保持对外接口不变")
    async for m in client.receive_response():
        if isinstance(m, UserMessage) and getattr(m, "uuid", None):
            checkpoints.append(m.uuid)            # ③ 收集 checkpoint ID
        if isinstance(m, ResultMessage):
            session_id = m.session_id             # 事后回滚要靠它定位 session

    # ④ 不认可 agent 的改动：把磁盘回滚到该点（本次新建的文件删除、改过的恢复内容）。
    #    回滚的是文件不是对话
    await client.rewind_files(checkpoints[0])

checkpointing 的边界：

- 只跟踪 `Write`/`Edit`/`NotebookEdit` 的改动（Bash 改的文件不算）；只回滚**文件内容**——目录的创建/移动/删除不回滚；只管本地文件，远程/网络文件不跟踪。
- checkpoint 绑定 session。`query()` 的流迭代完后连接已关闭，事后回滚要 resume 同一 session、发空 prompt 再调 `rewind_files`——且**执行 rewind 的这个 resumed session 也必须设 `enable_file_checkpointing=True`**，否则报 "File rewinding is not enabled"。
- 上方 cell 在同一个 `ClaudeSDKClient` 连接内事后回滚（context manager 未退出、连接仍开）；官方示例展示的是"流内即时回滚"和"resume 后回滚"两种形态，语义相同。

**任务清单**：agent 处理多步任务时会用 `TaskCreate` / `TaskUpdate` / `TaskGet` / `TaskList` 工具维护任务列表，监听对应 `ToolUseBlock` 就能给用户渲染实时进度条。监听要点：

- 切换点按 **CLI 版本**算（捆绑或 `cli_path` 指向的 Claude Code ≥ v2.1.142 用 Task 系工具），不是 Python 包版本；还想收旧的 `TodoWrite` 就设 `env={"CLAUDE_CODE_ENABLE_TASKS": "0"}`。`TodoWrite` 的 item 是 `{content, status, activeForm}`（`in_progress` 时渲染 `activeForm`），生命周期 `pending → in_progress → completed`、整组完成后移除。
- Task 系：`TaskCreate` 入参 `{subject, description, activeForm?, metadata?}`——**分配的 task ID 不在入参里**，在对应 `tool_result` 的 `{task: {id, subject}}` 中，要从那里取来做映射的 key；`TaskUpdate` 入参 `{taskId, status?, subject?, ...}`，`status: "deleted"` 表示删除。
- 流里的 `tool_use` input 是模型原始输出：Claude Code 执行前会把 `id`/`task_id` 修正为 `taskId`、`active_form` 修正为 `activeForm`，但**修正不反映在流里**——监听代码要防御性读取：`input.get("taskId") or input.get("id") or input.get("task_id")`。

### 21.5 Sandbox：给 Bash 套上笼子

`sandbox` 选项在 OS 层限制命令的文件系统与网络访问（Linux 依赖 `bubblewrap`/`socat`，macOS 用 `sandbox-exec`）：

```python
options = ClaudeAgentOptions(
    sandbox={
        "enabled": True,
        "autoAllowBashIfSandboxed": True,        # 沙箱内的 Bash 自动批准（默认 True）
        "network": {"allowedDomains": ["api.example.com"], "allowLocalBinding": True},
    }
)
```

关键语义（字段名 camelCase，直接映射 wire 格式——SDK 与 CLI 子进程之间传输的原始 JSON——的键名）：

- `excludedCommands`（如 `["docker"]`）：静态名单，**总是**绕过沙箱，模型无权干预。
- `allowUnsandboxedCommands`（默认 True）：允许模型在 tool input 里设 `dangerouslyDisableSandbox: True` 申请出沙箱——这类请求**回落到权限系统**，`can_use_tool` 会被调用，可以在回调里做审计和白名单。**`bypassPermissions` + 该开关 = 模型可静默逃逸沙箱**，别组合使用。
- 沙箱不可用时默认降级为无沙箱执行（stderr 警告）；设 `"failIfUnavailable": True` 改为报 `error_during_execution`。
- 网络 allowlist 只按 hostname 过滤、不做 TLS 检查，domain fronting 可能绕过；更强保证要上 TLS 终止代理。


### 21.6 部署：子进程模型决定一切

一个 session = 一个 `claude` 子进程；三类状态默认落在容器本地盘（session transcript、CLAUDE.md、工作目录产物），**容器重启即丢**。由此推出四种 session 形态：

| 形态 | 做法 | 适用 |
|---|---|---|
| Ephemeral | 每任务一容器，跑完即毁 | 一次性任务（修 bug、抽取、转换） |
| Long-running | 常驻容器 + `ClaudeSDKClient` 挂长会话 | 邮件 agent、聊天机器人 |
| Hybrid | 临时容器 + `session_store` 启动时补水 | 间歇性回访的长项目（**store 是必需不是可选**） |
| Multi-agent | 一容器多子进程，各配独立 `cwd` | 多 agent 协作模拟 |

资源与扩缩：起步 1 GiB RAM / 5 GiB 盘 / 1 CPU 每 agent；并发上限由内存决定——`每主机 agent 数 = (主机 RAM - 开销) / 单 session 峰值 RSS`；长会话容器用 `sessionId` 一致性哈希钉到固定实例。

已知限制四条：session 没有总超时（用 `max_turns` 兜底）；长会话内存增长（定期回收子进程）；宽扇出子 agent 会撞 API 限流（拆小批次）；**子 agent 没有 wall-clock 时长上限**——`CLAUDE_ASYNC_AGENT_STALL_TIMEOUT_MS` 只是卡死看门狗、不是总时长上限，给每个 `AgentDefinition` 设 `maxTurns` 兜底。

入站边界：认证放在 agent 容器前面的网关，agent 只接**预认证**的请求、不做用户 token 校验；`claude` 子进程本身不监听任何网络端口，入站面完全由你的应用层决定。

### 21.7 安全部署

威胁模型：prompt injection（agent 处理的内容里埋了指令）与模型失误。先认识两项**内建防线**（都是权限闸门，不是沙箱）：Bash 命令执行前会被解析成 AST 再对权限规则匹配（`eval` 等少数构造总是要求审批）；WebSearch 的结果先摘要化再进 context，降低网页 prompt injection 的风险。生产部署在此之上叠四层纵深防御：

1. **隔离**：sandbox runtime（轻量）→ 容器（`--cap-drop ALL` / `--read-only` / `--network none` + Unix socket 走代理）→ gVisor（拦截 syscall）→ microVM（Firecracker）。强度递增、开销递增。分界线是**内核**：sandbox 和容器与宿主共享内核，内核漏洞理论上可逃逸；要内核级隔离就上 gVisor 或独立 VM。
2. **凭证走代理**：agent 环境里不放 key。出站三条路——① `ANTHROPIC_BASE_URL` 指向代理，由它注入 Anthropic key；② 系统级 `HTTP_PROXY` / `HTTPS_PROXY`（局限：HTTPS 下代理只见 CONNECT 隧道、无法注入凭证）；③ TLS 终止代理（要件：agent 信任库里装代理的 CA 证书；注意**不是所有程序尊重 HTTP_PROXY**——Node.js 的 `fetch()` 默认忽略，Node 24+ 需 `NODE_USE_ENV_PROXY=1`；要全覆盖用 proxychains 或 iptables 透明代理）。其他服务的凭证也可走自定义 tool 转发到边界外执行。
3. **最小权限文件系统**：代码只读挂载；**`.env`、`~/.aws/credentials`、`~/.ssh`、`*.pem`、`~/.git-credentials`、`~/.kube/config`、`.npmrc` / `.pypirc`、`~/.docker/config.json` 等泄密文件挂载前剔除**；可写区用 tmpfs。
4. **审计**：代理记全部出站请求；多租户时在代理上做**逐租户出站策略**（独立出口 IP / 凭证 / 域名白名单），防止一个被攻陷的租户借别家的出站规则外传数据；OTel 的 `tool_decision` / `tool_result` 事件（§21.3「OpenTelemetry」）做逐用户审计流。

### 21.8 从 claude-code-sdk 迁移

旧包 `claude-code-sdk` → `claude-agent-sdk`，三处破坏性变化：包名与导入（`claude_code_sdk` → `claude_agent_sdk`）；`ClaudeCodeOptions` → `ClaudeAgentOptions`；**system prompt 不再默认 Claude Code 完整版**（要旧行为需显式 `system_prompt={"type": "preset", "preset": "claude_code"}`，见 §19「系统提示定制」）。`setting_sources` 默认值曾在 v0.1.0 短暂改为不加载、后已回退到全加载；Python ≤ 0.1.59 把 `setting_sources=[]` 当作省略处理，依赖空列表隔离前先升级。
